# NOMBRE : CESAR MAYTA

## PROYECTO : detectar toxicidad en comentarios online

La detección de toxicidad permite **identificar lenguaje ofensivo, insultos o discursos de odio** en redes sociales, foros o comentarios.

Es fundamental para:

- 🛡️ **Proteger comunidades online** (YouTube, Wikipedia, X, Reddit).  
- ⚙️ **Filtrar contenido automáticamente.**  
- 🤖 **Entrenar moderadores automáticos con IA.**

## Exploración del dataset Jigsaw Toxic Comment Classification

In [1]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
dataset = load_dataset("jhan21/jigsaw-toxic-comment-classification", split="train[:2%]")
dataset = dataset.shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.2)

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.csv:   0%|          | 0.00/68.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/159571 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 2552
    })
    test: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 639
    })
})

## convertimos el dataset a DataFrames de pandas

In [2]:
import pandas as pd

# Convert the train dataset to a pandas DataFrame for easier analysis
train_df = dataset["train"].to_pandas()

# Get the category columns
category_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Calculate the number of comments per category
category_counts = train_df[category_cols].sum()

print("Número de comentarios por categoría en el conjunto de entrenamiento:")
print(category_counts)

Número de comentarios por categoría en el conjunto de entrenamiento:
toxic            264
severe_toxic      32
obscene          141
threat            10
insult           144
identity_hate     27
dtype: int64


# Configuración y Entrenamiento del Modelo BERT

## 📘 Carga del modelo preentrenado

In [3]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

model_name = "distilbert-base-uncased"
model = DistilBertForSequenceClassification.from_pretrained(model_name)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## 📘 Preparación de DataLoader y pipeline de entrenamiento

In [4]:
def tokenize(batch):
    return tokenizer(batch["comment_text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("toxic", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]


Map:   0%|          | 0/2552 [00:00<?, ? examples/s]

Map:   0%|          | 0/639 [00:00<?, ? examples/s]

## Entrenamiento de Trainer con Hugging Face

In [18]:
from transformers import TrainingArguments, Trainer

new_model_name = "cesarcodigo-toxicidad"
# Configurar entrenamiento
training_args = TrainingArguments(
    output_dir=f"./{new_model_name}",          # Directorio de salida corregido
    num_train_epochs=3,                       # Número total de épocas de entrenamiento
    per_device_train_batch_size=16,           # Tamaño del batch por dispositivo durante el entrenamiento
    per_device_eval_batch_size=64,            # Tamaño del batch para evaluación
    warmup_steps=500,                         # Número de pasos de calentamiento para el scheduler de tasa de aprendizaje
    weight_decay=0.01,                        # Fuerza de la penalización L2
    logging_dir=f"./logs_{new_model_name}",   # Directorio para almacenar logs
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Step,Training Loss
10,0.006700
20,0.010200
30,0.002800
40,0.013100
50,0.001700
60,0.047300
70,0.018800
80,0.003900
90,0.001100
100,0.001600


TrainOutput(global_step=480, training_loss=0.017943862832180458, metrics={'train_runtime': 84.887, 'train_samples_per_second': 90.191, 'train_steps_per_second': 5.655, 'total_flos': 253542601027584.0, 'train_loss': 0.017943862832180458, 'epoch': 3.0})

In [ ]:
trainer.evaluate()

## Prueba del modelo con un texto de ejemplo

In [11]:
import torch

# Texto de ejemplo tóxico
toxic_text = "You are an idiot and your comments are stupid!"

# Tokenizar el texto
inputs = tokenizer(toxic_text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")

# Mover los inputs al mismo dispositivo que el modelo (GPU si está disponible, CPU en caso contrario)
if torch.cuda.is_available():
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Realizar la predicción
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Obtener las probabilidades (logits) y la clase predicha
logits = outputs.logits
probabilities = torch.softmax(logits, dim=1)
predicted_class_id = torch.argmax(logits, dim=1).item()

# Interpretar el resultado (clase 0: no tóxico, clase 1: tóxico)
prediction_label = "Tóxico" if predicted_class_id == 1 else "No Tóxico"

print(f"Texto de entrada: '{toxic_text}'")
print(f"Logits: {logits.cpu().numpy()}")
print(f"Probabilidades: {probabilities.cpu().numpy()}")
print(f"Clase predicha (0=No tóxico, 1=Tóxico): {predicted_class_id}")
print(f"Resultado: {prediction_label}")

Texto de entrada: 'You are an idiot and your comments are stupid!'
Logits: [[-2.4811847  2.2222383]]
Probabilidades: [[0.00898278 0.9910172 ]]
Clase predicha (0=No tóxico, 1=Tóxico): 1
Resultado: Tóxico


# GUARDAMOS EL MODELO

In [ ]:
trainer.save_model(new_model_name)
tokenizer.save_pretrained(new_model_name)

In [ ]:
!hf auth login

# PUBLICAMOS EL MODELO EN HUGGING FACE

In [ ]:
trainer.push_to_hub(new_model_name)

print(f"¡El modelo ha sido re-publicado exitosamente en Hugging Face con el nombre '{new_model_name}'!")
print(f"Revisa tu perfil en Hugging Face para confirmar el nuevo repositorio: https://huggingface.co/cesarcodigo/{new_model_name}")